# PupilSense Left-Eye Reproduction (Stage 1)

Runs the released ResNet18/ResNet50 checkpoints over the EyeDentify dataset to reproduce the left-eye MAE/MAPE from *PupilSense* (Shah et al., ETRA '25).

**Before running:** in the Colab menu, go to Runtime > Change runtime type and select a GPU.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Get the package code from GitHub (public) so it always matches this notebook.
# Data + weights still come from Drive. Re-run this cell to pull the latest code.
!rm -rf /content/EyeBiomarkers
!git clone -q https://github.com/ShirleyMgit/EyeBiomarkers.git /content/EyeBiomarkers
PACKAGE_PARENT = "/content/EyeBiomarkers"
print("Cloned package ->", PACKAGE_PARENT)

## Stage data to local disk (do this — it's the difference between minutes and hours)

The dataset is ~212k tiny PNGs; reading them over the Google Drive mount is very slow (a network round-trip per file). This cell copies the single `left_eyes_data.zip` to Colab's local SSD and unzips it there, so inference reads from local disk.

In [ ]:
# Stage the dataset to Colab local disk (fast) instead of reading 200k+ tiny files over the Drive mount (very slow).
from pathlib import Path
import glob

ZIP_ON_DRIVE = "/content/drive/MyDrive/pupilsense_data_code/data/left_eyes_data.zip"  # single left-eye archive

!cp "{ZIP_ON_DRIVE}" /content/left_eyes_data.zip
!rm -rf /content/data && mkdir -p /content/data
!unzip -q -o /content/left_eyes_data.zip -d /content/data

_csvs = glob.glob("/content/data/**/session_data.csv", recursive=True)
assert _csvs, "No session_data.csv found after unzip - check ZIP_ON_DRIVE path / zip contents."
LOCAL_DATA_ROOT = Path(_csvs[0]).parent.parent.parent  # .../<root>/<participant>/<session>/session_data.csv
print("Sessions unzipped:", len(_csvs), "| LOCAL_DATA_ROOT =", LOCAL_DATA_ROOT)

import sys
from pathlib import Path
import torch

# Code comes from the GitHub clone above (/content/EyeBiomarkers). Weights come from Drive.
PACKAGE_PARENT = globals().get("PACKAGE_PARENT", "/content/EyeBiomarkers")
WEIGHTS_DIR = Path("/content/drive/MyDrive/pupilsense_data_code/code/pupilsense/pre_trained_models")  # contains ResNet18/ and ResNet50/ (each with left_eye.pt)

# DATA_ROOT comes from the staging cell (local SSD). Recover it if that variable isn't set yet.
if "LOCAL_DATA_ROOT" in globals():
    DATA_ROOT = LOCAL_DATA_ROOT
else:
    _staged = sorted(Path("/content/data").glob("**/session_data.csv"))
    if not _staged:
        raise RuntimeError("Run the 'Stage data to local disk' cell above first (it unzips the dataset to /content/data).")
    DATA_ROOT = _staged[0].parent.parent.parent

sys.path.insert(0, PACKAGE_PARENT)
from pupilsense_repro.config import ReproConfig
from pupilsense_repro.runner import run_reproduction
from pupilsense_repro import plots

config = ReproConfig(
    data_root=DATA_ROOT,
    weights_dir=WEIGHTS_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_workers=2,            # parallel image loading (Colab has ~2 CPUs); big speedup off local disk
    results_dir=Path("/content/results"),
    figures_dir=Path("/content/figures"),
)
print("device:", config.device, "| data_root:", config.data_root)

In [ ]:
import sys
from pathlib import Path
import torch

# Paths preset for this Drive layout. Edit only if you move the folders.
PACKAGE_PARENT = "/content/drive/MyDrive/pupilsense_data_code/code_claude/EyeBiomarkers"              # folder that CONTAINS pupilsense_repro/
WEIGHTS_DIR = Path("/content/drive/MyDrive/pupilsense_data_code/code/pupilsense/pre_trained_models")  # contains ResNet18/ and ResNet50/ (each with left_eye.pt)

# DATA_ROOT comes from the staging cell (local SSD). Recover it if that variable isn't set yet.
if "LOCAL_DATA_ROOT" in globals():
    DATA_ROOT = LOCAL_DATA_ROOT
else:
    _staged = sorted(Path("/content/data").glob("**/session_data.csv"))
    if not _staged:
        raise RuntimeError("Run the 'Stage data to local disk' cell above first (it unzips the dataset to /content/data).")
    DATA_ROOT = _staged[0].parent.parent.parent

sys.path.insert(0, PACKAGE_PARENT)
from pupilsense_repro.config import ReproConfig
from pupilsense_repro.runner import run_reproduction
from pupilsense_repro import plots

config = ReproConfig(
    data_root=DATA_ROOT,
    weights_dir=WEIGHTS_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_workers=2,            # parallel image loading (Colab has ~2 CPUs); big speedup off local disk
    results_dir=Path("/content/results"),
    figures_dir=Path("/content/figures"),
)
print("device:", config.device, "| data_root:", config.data_root)

In [ ]:
# ResNet50 is the paper's better model (left-eye MAPE 3.23% vs ResNet18 3.41%).
# To also run ResNet18 for comparison, use bases=("resnet50", "resnet18").
summary = run_reproduction(config, bases=("resnet50",))
summary

In [ ]:
import pandas as pd
config.figures_dir.mkdir(parents=True, exist_ok=True)
per_part = pd.read_csv(config.results_dir / "per_participant_mape.csv", index_col=0)
per_participant_by_base = {b: per_part[b].dropna() for b in per_part.columns}

plots.plot_per_participant_mape(per_participant_by_base, config.figures_dir / "per_participant_mape.png")
plots.plot_mape_histogram(per_participant_by_base, config.figures_dir / "mape_hist.png")
for base in per_participant_by_base:
    preds = pd.read_csv(config.results_dir / f"predictions_{base}.csv")
    plots.plot_pred_vs_true(preds, config.figures_dir / f"pred_vs_true_{base}.png")
    plots.plot_diameter_over_frames(preds, participant_id=1, session_id=1,
                                    out_path=config.figures_dir / f"series_{base}.png")

from IPython.display import Image as IPyImage, display
display(IPyImage(str(config.figures_dir / "per_participant_mape.png")))

## Reading the results

`summary` (and `left_eye_metrics.csv` in `config.results_dir`) has one row per base architecture with `overall_mae`, `overall_mape`, and per-fold metrics; `per_participant_mape.csv` gives the per-participant MAPE distribution plotted above.

**Caveat:** the released weights are a single deployed checkpoint per eye/architecture, not per-fold models. Overall MAPE can be optimistic where the checkpoint trained on a participant that also appears in evaluation (train/test overlap). For a fairer comparison to the paper's left-eye numbers (ResNet18 â‰ˆ 3.41%, ResNet50 â‰ˆ 3.23%), look at the per-fold rows rather than the overall figure.